# Genotype-informed GNN for differential GRN inference (B73 x PT RILs)

Decisions baked in (all changeable in CFG):
  log-CPM expression; signs = -1 for antho/necro/wilt (higher score = more damage);
  Eq. 1 contrast = mean P of bottom/top 10% of lines; Z = TF expression until genotype
  data arrive; strict 1:1 v4->v5 mapping; nodes = network genes; reverse edges on;
  edge feature = e_ij(p) only (+ direction flag); node features = expression + is_TF.

In [1]:
from __future__ import annotations
import json
import math
import re
import time
from pathlib import Path
import numpy as np
import pandas as pd
# Define all the parameters for the entire project. 
# The parameters are adjusted for the ablation studies
CFG = {
    "raw_dir": "/kaggle/input/",             
    "out_dir": "results",
    "files": {
        "counts": "plant_counts_raw_UMI.csv",
        "meta": "plant_metadata.csv",
        "pheno": "Phenotyping_scores_manual_-_All_runs.csv",
        "mgrn": "mGRN_leaf.csv",
        "idmap": "B73v4_to_B73v5.tsv",
    },
    # data preprocessing
    "min_cpm": 1.0, "min_frac": 0.5, "min_umis": 200_000,
    # phenotype
    "traits": ["antho", "necro", "wilt"],
    "signs": {"antho": -1, "necro": -1, "wilt": -1},
    "conditions": ["composite"],
    "tail_frac": 0.10,
    # edge features
    "z_modes": ["genotype","tf_expression"],
    "min_genotyped_lines": 30,
    # graph
    "split": (0.7, 0.1, 0.2), "k_train_negatives": 5, "reverse_edges": True,
    # model and training parameters
    "hidden": 64, "dropout": 0.3, "lr": 1e-3, "weight_decay": 1e-4,
    "epochs": 400, "patience": 30, "p_mode": "sample",
    "seeds": [0, 1, 2],
    # differential
    "n_permutations": 20,
    # baselines for Random Forest
    "rf_tail_frac": 0.30,               
    "rf_max_targets": 1000,             
    "device": "cuda",                   
}
# assign variable to the input directory and output directory, create output dir
RAW, OUT = Path(CFG["raw_dir"]), Path(CFG["out_dir"])
OUT.mkdir(parents=True, exist_ok=True)
LOG: list[str] = []


def log(msg: str):
    print(msg)
    LOG.append(msg)

In [2]:
import glob
from pathlib import Path
# update the file paths since sometimes the data folder name changes when uploading to Kaggle
for key, name in CFG["files"].items():
    hits = glob.glob(f"/kaggle/input/**/{name}", recursive=True)
    if hits:
        CFG["files"][key] = hits[0]
    print(f"{key:7s} {'OK     ' if hits else 'MISSING'} {hits[0] if hits else name}")

CFG["out_dir"] = "/kaggle/working/results"

counts  OK      /kaggle/input/datasets/mariapugliese/gnn-data/plant_counts_raw_UMI.csv
meta    OK      /kaggle/input/datasets/mariapugliese/gnn-data/plant_metadata.csv
pheno   OK      /kaggle/input/datasets/mariapugliese/gnn-data/Phenotyping_scores_manual_-_All_runs.csv
mgrn    OK      /kaggle/input/datasets/mariapugliese/gnn-input/mGRN_leaf.csv
idmap   OK      /kaggle/input/datasets/mariapugliese/gnn-input/B73v4_to_B73v5.tsv


In [3]:
import glob
from pathlib import Path

CFG["files"].update({
    "vcf": "b74_x_pt_rils_nils_dh.vcf",            # matches b73_x_pt_rils_nils_dh.vcf.gz or ..._vcf.gz
    "sample_key": "sample_key.tsv",
    "marker_map": "marker_map.tsv",
})
CFG["z_mode"] = "genotype"

for key, name in CFG["files"].items():
    hits = [h for h in glob.glob(f"/kaggle/input/**/{Path(name).name}", recursive=True)
            if not h.endswith(".tbi")]
    if hits:
        CFG["files"][key] = hits[0]
    print(f"{key:11s} {'OK     ' if hits else 'MISSING'} {hits[0] if hits else name}")

counts      OK      /kaggle/input/datasets/mariapugliese/gnn-data/plant_counts_raw_UMI.csv
meta        OK      /kaggle/input/datasets/mariapugliese/gnn-data/plant_metadata.csv
pheno       OK      /kaggle/input/datasets/mariapugliese/gnn-data/Phenotyping_scores_manual_-_All_runs.csv
mgrn        OK      /kaggle/input/datasets/mariapugliese/gnn-input/mGRN_leaf.csv
idmap       OK      /kaggle/input/datasets/mariapugliese/gnn-input/B73v4_to_B73v5.tsv
vcf         OK      /kaggle/input/datasets/mariapugliese/gnn-vcf/b74_x_pt_rils_nils_dh.vcf
sample_key  OK      /kaggle/input/datasets/mariapugliese/gnn-genotype2/sample_key.tsv
marker_map  OK      /kaggle/input/datasets/mariapugliese/gnn-genotype2/marker_map.tsv


In [4]:
# Phenotype runs (1-6) map onto RNA data (inferred from chamber position + line)
SESSION_TO_PHENO_RUN = {"1_1": 1, "2_1": 2, "3_1": 3, "3_2": 4, "2_2": 5, "1_2": 6}

# take each RIL name and assign them to a name that matches, so they match in each file
def canonical_line(name) -> str:
    """LANPT16B023 / RIL023 -> RIL023; LANPT16BNIL084DH / NIL084DH -> NIL084DH; B73_05 -> B73."""
    s = str(name)
    if s.upper().startswith("B73"):
        return "B73"
    m = re.match(r"^(?:LANPT16B)?(NIL|RIL)?0*(\d+)(DH)?$", s, flags=re.I)
    if not m:
        return s
    return f"{(m.group(1) or 'RIL').upper()}{int(m.group(2)):03d}{(m.group(3) or '').upper()}"

# similar function but it parses through the phenotype of each sample
def parse_pheno_sample(sample):
    """'RUN1_A10_RIL181.tif' -> (1, 'A', 10, 'RIL181'); tolerates 'Run2' and a broken '.tif'."""
    m = re.match(r"^run(\d+)_([A-H])(\d+)_([A-Za-z0-9]+?)(?:\.t\w*)?$", str(sample), flags=re.I)
    if not m:
        return (np.nan, None, np.nan, None)
    return (int(m.group(1)), m.group(2).upper(), int(m.group(3)), canonical_line(m.group(4)))

# %% Genotype data — add to CFG["files"] and paste these functions into the pipeline
# CFG["files"].update({"vcf": "b73_x_pt_rils_nils_dh_vcf.gz",
#                      "sample_key": "sample_key.tsv",
#                      "marker_map": "marker_map.tsv"})
# CFG["z_mode"] = "genotype"
import gzip

import numpy as np
import pandas as pd

GT_CODE = {"0/0": 0.0, "0/1": 1.0, "1/0": 1.0, "1/1": 2.0, "0|0": 0.0, "0|1": 1.0, "1|0": 1.0, "1|1": 2.0}


import gzip

def open_maybe_gzip(path):
    """Open a text file whether or not it's gzip-compressed (Kaggle may decompress uploads)."""
    with open(path, "rb") as fh:
        is_gz = fh.read(2) == b"\x1f\x8b"
    return gzip.open(path, "rt") if is_gz else open(path, "rt")


def read_vcf_dosage(path) -> pd.DataFrame:
    """Samples x marker genes, PT allele dosage from GT (0 = B73/B73, 1 = het, 2 = PT/PT, NaN = missing)."""
    genes, rows, samples = [], [], None
    with open_maybe_gzip(path) as f:
        for line in f:
            if line.startswith("##"):
                continue
            p = line.rstrip("\n").split("\t")
            if line.startswith("#CHROM"):
                samples = p[9:]
                continue
            genes.append(p[2])
            rows.append([GT_CODE.get(x.split(":")[0], np.nan) for x in p[9:]])
    return pd.DataFrame(np.array(rows, dtype=float).T, index=samples, columns=genes)
def load_genotypes():
    """Returns dosage (samples x markers), sample key (unit_id -> line, class), marker map."""
    f = CFG["files"]
    dos = read_vcf_dosage(RAW / f["vcf"])
    key = pd.read_csv(RAW / f["sample_key"], sep="\t").set_index("unit_id")
    key["line"] = key["line"].map(canonical_line)
    mmap = pd.read_csv(RAW / f["marker_map"], sep="\t")
    log(f"Genotypes: {dos.shape[0]} samples x {dos.shape[1]} marker genes; "
        f"missing {np.isnan(dos.values).mean():.1%}")
    return dos, key, mmap


def verified_labels(meta: pd.DataFrame, key: pd.DataFrame) -> pd.Series:
    """Line label for each RNA sample, preferring the genotype-verified sample key.

    The RNA metadata's line labels are wrong for plate PN3 (0 of 66 genotyped RIL samples
    match the key; the key's labels are confirmed by replicate-plant genotype concordance,
    97% vs 90%, and by PT fractions of NIL/B73-labelled samples). Rule:
      - sample genotyped  -> line from the sample key
      - not genotyped, plate PN3 -> dropped (identity can't be verified)
      - not genotyped, other plates -> RNA metadata label (these plates agree with the key)
    """
    plate = meta["sample_id"].str.split("_").str[0]
    lab = meta["sample_id"].map(key["line"])
    from_meta = lab.isna() & (plate != "PN3")
    lab[from_meta] = meta.loc[from_meta, "genotype"].map(canonical_line)
    disagree = lab.notna() & ~from_meta & (lab != meta["genotype"].map(canonical_line))
    log(f"Sample labels: {(~from_meta & lab.notna()).sum()} from genotype key "
        f"({disagree.sum()} differ from RNA metadata, mostly plate PN3); "
        f"{from_meta.sum()} from metadata; {lab.isna().sum()} PN3 samples dropped (unverifiable)")
    return lab


def genotype_line_matrix(dos, key, lines) -> pd.DataFrame:
    """Lines x marker genes, PT allele fraction (dosage / 2), averaged over a line's genotyped plants."""
    ril = key.index[key["class"] == "ril"]
    d = dos.loc[dos.index.intersection(ril)]
    Zm = (d / 2.0).groupby(key.loc[d.index, "line"]).mean()
    Zm = Zm.reindex(lines)
    log(f"Genotype matrix: {Zm.notna().any(axis=1).sum()} of {len(lines)} lines genotyped; "
        f"mean PT fraction {np.nanmean(Zm.values):.3f} (BC1S5 expectation 0.25)")
    return Zm


def assign_genes_to_markers(genes, mmap) -> pd.Series:
    """Map every gene to a marker gene. A gene that is itself a marker maps to itself; otherwise
    to the nearest marker by v5 ID number within its chromosome. v5 IDs are numbered along each
    chromosome (Spearman rho = 1.00 with position; ~1% local inversions), so ID distance is a
    proxy for physical distance. Replace with GFF coordinates if exact positions are needed."""
    num = lambda s: s.str.extract(r"eb(\d+)", expand=False).astype(float)
    m = mmap.assign(num=num(mmap.gene_id)).sort_values("num")
    bounds = m.groupby("chrom").num.agg(["min", "max"])
    g = pd.Series(genes, index=genes)
    gnum = num(g)
    out = pd.Series(index=genes, dtype=object)
    is_marker = g.isin(set(m.gene_id))
    out[is_marker] = g[is_marker]
    for chrom, (lo, hi) in bounds.iterrows():
        mc = m[m.chrom == chrom]
        sel = ~is_marker & (gnum >= lo - 5000) & (gnum <= hi + 5000) & out.isna()
        if sel.any():
            idx = np.searchsorted(mc.num.values, gnum[sel].values)
            idx = np.clip(idx, 1, len(mc) - 1)
            left, right = mc.num.values[idx - 1], mc.num.values[idx]
            pick = np.where(gnum[sel].values - left <= right - gnum[sel].values, idx - 1, idx)
            out[sel] = mc.gene_id.values[pick]
    log(f"Genes mapped to markers: {is_marker.sum()} are markers, "
        f"{(~is_marker & out.notna()).sum()} use nearest marker, {out.isna().sum()} unmapped")
    return out


def load_genotype_Z(expr_line: pd.DataFrame) -> pd.DataFrame:
    """Lines x genes: PT allele fraction at each gene's locus (0 = B73, 1 = PT, 0.5 = het).
    Uses GENO (set by prepare_data) so the VCF is read once."""
    Zm, gene2marker = GENO["Z_markers"], GENO["gene2marker"]
    genes = [g for g in expr_line.columns if pd.notna(gene2marker.get(g))]
    Z = Zm.reindex(index=expr_line.index)[gene2marker[genes].values]
    Z.columns = genes
    return Z


# %% Replacement for prepare_data (line labels from genotypes; phenotype by line name)
GENO = {}


def prepare_data():
    f = CFG["files"]
    counts = pd.read_csv(RAW / f["counts"], index_col="plant_id")
    meta = pd.read_csv(RAW / f["meta"])
    pheno = pd.read_csv(RAW / f["pheno"])
    log(f"Raw counts: {counts.shape[0]} plants x {counts.shape[1]} genes")

    dos, key, mmap = load_genotypes()
    meta["line"] = verified_labels(meta, key).values
    meta = meta.set_index("plant_id")

    # phenotype: line means over ALL phenotyped plants of a line, matched by line name.
    # (Plant-level position joins can't be trusted for plate PN3, whose sample->plant map is wrong.)
    parsed = pd.DataFrame(pheno["Sample"].map(parse_pheno_sample).tolist(),
                          columns=["pheno_run", "row", "col", "pheno_line"], index=pheno.index)
    pheno = pd.concat([pheno, parsed], axis=1).dropna(subset=["pheno_line"])
    pheno_by_line = pheno.groupby("pheno_line")[CFG["traits"]].mean()

    low = meta["total_UMIs_kept_genes"] < CFG["min_umis"]
    is_ril = meta["line"].fillna("").str.startswith("RIL")
    keep = is_ril & ~low & meta["line"].isin(pheno_by_line.dropna().index)
    kept = meta.index[keep]
    log(f"RIL plants kept: {len(kept)} from {meta.loc[kept, 'line'].nunique()} lines")

    cpm_raw = counts.div(counts.sum(axis=1), axis=0) * 1e6
    genes = cpm_raw.columns[(cpm_raw >= CFG["min_cpm"]).mean(axis=0) >= CFG["min_frac"]]
    lib = meta.loc[counts.index, "total_UMIs_kept_genes"] * meta.loc[counts.index, "tmm_norm_factor"]
    expr = np.log2(counts[genes].div(lib, axis=0) * 1e6 + 1).loc[kept]
    line = meta.loc[kept, "line"]
    expr_line = expr.groupby(line).mean()
    pheno_line = pheno_by_line.loc[expr_line.index]

    GENO["Z_markers"] = genotype_line_matrix(dos, key, expr_line.index)
    GENO["gene2marker"] = assign_genes_to_markers(list(genes), mmap)
    log(f"Genes kept: {len(genes)}; line-level expression {expr_line.shape}")
    log("Line-level trait Spearman:\n" + pheno_line.corr("spearman").round(2).to_string())
    return expr_line, pheno_line, meta

In [5]:
# create the phenotype vector, P
# align the signs to represent tolerance as a high score and low tolerance as a lower score
# depending on the condition type, either average the phenotypic scores or 
def get_P(pheno_line: pd.DataFrame, condition: str) -> pd.Series:
    """Sign-aligned (+ = tolerant), z-scored P. The composite is re-standardized after
    averaging so beta3 / Delta are on the same scale for every condition."""
    traits = CFG["traits"]
    trait_p = pheno_line[traits].astype(float)
    # get the z-score of each trait, then flip the sign
    trait_p = (trait_p - trait_p.mean()) / trait_p.std(ddof=0) * pd.Series(CFG["signs"])[traits]
    # adjust what P is based off of the predefined parameters inf CFG
    # otherwise, use the first principal component PC1
    if condition == "composite":
        P = trait_p.mean(axis=1)
    elif condition in traits:
        P = trait_p[condition]
    else:
        raise ValueError(condition)
    # standardize the final P, z-scoring the P values again
    return ((P - P.mean()) / P.std(ddof=0)).rename(condition)

# this function identifies the top and bottom performing RILs for cold tolerance,
# given the phenotype score
def tail_lines(P: pd.Series, frac: float, seed: int = 0):
    # find the bottom and top lines using phenotype scores
    k = math.ceil(frac * len(P))
    # randomized tie breaker since many RILs have the same score
    tb = pd.Series(np.random.default_rng(seed).permutation(len(P)), index=P.index)
    order = pd.DataFrame({"P": P, "tb": tb}).sort_values(["P", "tb"]).index
    return order[:k], order[-k:]

# take the average of the low and high tolerance RILs
def contrast_values(P: pd.Series, frac: float):
    lo, hi = tail_lines(P, frac)
    return float(P.loc[lo].mean()), float(P.loc[hi].mean())

In [6]:
# create the Z matrix, using expression data. This is the z-scores for each line
def make_Z(expr_line: pd.DataFrame) -> pd.DataFrame:
    """Z_j for each TF: genotype (PT allele fraction at the locus) or standardized expression."""
    mode = CFG["z_mode"]
    if mode == "tf_expression":
        return (expr_line - expr_line.mean()) / expr_line.std(ddof=0).replace(0, np.nan)
    if mode == "genotype":
        return load_genotype_Z(expr_line)
    raise ValueError(f"Unknown z_mode {mode!r}; use 'genotype' or 'tf_expression'")
    
# fit a linear regression to find b3, the genotype x phenotype metric
# X is the lines and genes after log-CPM transformation
# P is the phenotype vector
# Z is the lines x gene values for each transcription factor (TF). Right now, it's just standardized
# expression
# the pairs list is a table of tf,target pairs
# everything is centered and normalized to std and variance of (0, 1)
def fit_gxp(X: pd.DataFrame, Z: pd.DataFrame, P: pd.Series, pairs: pd.DataFrame,
            min_lines: int = 30) -> pd.DataFrame:
    """Eq. 2 for every (tf, target) pair: X_i ~ b0 + b1 Z_j + b2 P + b3 Z_j P.
    Targets standardized, P centered. One least-squares solve per TF (all its targets
    share the design matrix). Verified against statsmodels OLS to 6 decimals."""
    # align the X, P, and Z matrices
    lines = X.index.intersection(P.index).intersection(Z.index)
    # variable to record the lines
    Xv = X.loc[lines]
    # z-score each target gene across the lines
    Xa = ((Xv - Xv.mean()) / Xv.std(ddof=0).replace(0, np.nan)).to_numpy(float)
    # lookup table from gene name to column number, to pull target columns out of the array
    col = {g: i for i, g in enumerate(Xv.columns)}
    # centers P at zero
    p = P.loc[lines].to_numpy(float)
    p = p - p.mean()
    # drops pairs that can't be fitted, such as a TF pairing with itself, a TF with no Z
    # column, or target genes not in the expression matrix
    Za = Z.loc[lines]
    pairs = pairs[(pairs.tf != pairs.target) & pairs.tf.isin(Za.columns) & pairs.target.isin(col)]
    out = []
    # one regression per TF, looping over the TFs
    for tf, grp in pairs.groupby("tf", sort=False):
        # z is TF j's value in each line, TF is skipped if less than 30 lines are usable
        # or if Z doesn't vary
        z = Za[tf].to_numpy(float)
        ok = ~np.isnan(z)
        if ok.sum() < min_lines or np.nanstd(z) == 0:
            continue
        # matrix with one row = one line, with four columns matching the coefficients
        # first one is for the intercept, second for TF value, third for phenotype,
        # fourth for the interaction score between genotype and phenotype (expression for now)
        D = np.column_stack([np.ones(ok.sum()), z[ok], p[ok], z[ok] * p[ok]])
        Y = Xa[np.ix_(ok, grp.target.map(col).to_numpy())]
        # ordinary least squares for all targets at once. B has four rows and one column
        # per target, lstsq returns other outputs
        B, *_ = np.linalg.lstsq(D, Y, rcond=None)
        # define the residual variance for b3, the interaction score, then get standard error
        sigma2 = ((Y - D @ B) ** 2).sum(0) / (ok.sum() - 4)
        se3 = np.sqrt(sigma2 * np.linalg.pinv(D.T @ D)[3, 3])
        # also include the t-statistic, which is how strong the interaction is relative to noise
        out.append(pd.DataFrame({"tf": tf, "target": grp.target.to_numpy(),
                                 "b1": B[1], "b3": B[3], "t_b3": B[3] / se3}))
    res = pd.concat(out, ignore_index=True)
    res.attrs["p_center"] = float(P.loc[lines].mean())
    return res

In [7]:
SPLITS = ("train", "val", "test")

# 
def load_network(expressed: pd.Index) -> pd.DataFrame:
    f = CFG["files"]
    m = pd.read_csv(RAW / f["idmap"], sep="\t", header=None, names=["v4", "v5"], dtype=str).dropna()
    m["v5s"] = m.v5.str.split(",").map(lambda xs: [x.strip() for x in xs if x.strip()])
    per_v5 = m.explode("v5s").groupby("v5s").v4.nunique()
    one = m[m.v5s.str.len() == 1].assign(v5one=lambda d: d.v5s.str[0])
    one = one[~one.v5one.isin(per_v5[per_v5 > 1].index)]
    id_map = dict(zip(one.v4, one.v5one))
    log(f"ID map: {len(id_map):,} strict 1:1 v4->v5 pairs")

    d = pd.read_csv(RAW / f["mgrn"])
    d = d[~d.Target.str.startswith("ENSRNA")]
    log(f"mGRN leaf: {len(d):,} edges, {d.TF.nunique():,} TFs")
    e = pd.DataFrame({"tf": d.TF.map(id_map), "target": d.Target.map(id_map)}).dropna()
    log(f"  after v4->v5: {len(e):,} edges, {e.tf.nunique():,} TFs")
    e = e[e.tf.isin(expressed) & e.target.isin(expressed) & (e.tf != e.target)].drop_duplicates()
    log(f"  after expressed-gene filter: {len(e):,} edges, {e.tf.nunique():,} TFs, "
        f"{e.target.nunique():,} targets")
    return e.reset_index(drop=True)


def split_by_tf(edges: pd.DataFrame, seed: int) -> pd.Series:
    """Whole TFs to train/val/test, stratified by out-degree quartile, cut by edge fraction."""
    rng = np.random.default_rng(seed)
    deg = edges.tf.value_counts()
    strata = pd.qcut(deg.rank(method="first"), 4, labels=False)
    cuts = np.cumsum(CFG["split"])[:-1]
    assign = {}
    for s in range(4):
        tfs = deg.index[strata == s].to_numpy()
        tfs = tfs[rng.permutation(len(tfs))]
        cum = np.cumsum(deg.loc[tfs].to_numpy()) / deg.loc[tfs].sum()
        for tf, k in zip(tfs, np.searchsorted(cuts, cum, side="left")):
            assign[tf] = SPLITS[min(k, 2)]
    return pd.Series(assign)


def sample_negatives(pos_tf, n, prob, forbidden, k, rng):
    """Corrupt targets: same TF, target ~ in-degree, never a known positive or self-pair."""
    tf = np.repeat(pos_tf, k)
    tg = rng.choice(n, size=len(tf), p=prob)
    for _ in range(100):
        bad = (tg == tf) | np.isin(tf.astype(np.int64) * n + tg, forbidden)
        if not bad.any():
            return tg
        tg[bad] = rng.choice(n, size=bad.sum(), p=prob)
    raise RuntimeError("negative sampling did not converge")


def build_graph(expr_line, edges, seed):
    genes = pd.Index(sorted(set(edges.tf) | set(edges.target)))
    gidx = pd.Series(np.arange(len(genes)), index=genes)
    n = len(genes)
    Xz = expr_line[genes]
    Xz = ((Xz - Xz.mean()) / Xz.std(ddof=0).replace(0, np.nan)).fillna(0)
    is_tf = genes.isin(edges.tf.unique()).astype(np.float32)
    x = np.column_stack([Xz.T.to_numpy(np.float32), is_tf])

    split = split_by_tf(edges, seed)
    tf_i, tg_i = gidx.loc[edges.tf].to_numpy(), gidx.loc[edges.target].to_numpy()
    forbidden = np.unique(tf_i.astype(np.int64) * n + tg_i)
    indeg = np.bincount(tg_i, minlength=n).astype(float)
    rng = np.random.default_rng(seed + 1000)
    es = edges.tf.map(split).to_numpy()
    q = {}
    for s in SPLITS:
        m = es == s
        k = CFG["k_train_negatives"] if s == "train" else 1
        neg = sample_negatives(tf_i[m], n, indeg / indeg.sum(), forbidden, k, rng)
        q[s] = {"pairs": np.vstack([np.column_stack([tf_i[m], tg_i[m]]),
                                    np.column_stack([np.repeat(tf_i[m], k), neg])]),
                "label": np.r_[np.ones(m.sum()), np.zeros(m.sum() * k)].astype(np.float32),
                "slot": np.r_[-np.ones(m.sum(), int), np.tile(np.arange(k), m.sum())]}
        log(f"  {s}: {(split == s).sum()} TFs, {m.sum():,} positives, {m.sum() * k:,} negatives")

    # message passing: train positives (+ reverse copies unless that reverse pair is held out)
    trpos = q["train"]["pairs"][q["train"]["label"] == 1]
    mp, mp_dir = [trpos], [np.zeros(len(trpos))]
    if CFG["reverse_edges"]:
        held = np.concatenate([q[s]["pairs"][q[s]["label"] == 1] for s in ("val", "test")])
        held_codes = held[:, 0].astype(np.int64) * n + held[:, 1]
        rev = trpos[:, ::-1]
        keep = ~np.isin(rev[:, 0].astype(np.int64) * n + rev[:, 1], held_codes)
        mp.append(rev[keep])
        mp_dir.append(np.ones(keep.sum()))
        log(f"  reverse edges dropped (reciprocal pair held out): {(~keep).sum()}")
    G = {"genes": genes, "x": x, "is_tf": is_tf, "q": q,
         "mp_pairs": np.vstack(mp), "mp_dir": np.concatenate(mp_dir).astype(np.float32),
         "mp_fwd_pairs": trpos}
    check_leakage(G, n)
    return G


def check_leakage(G, n):
    q = G["q"]
    code = lambda pr: set((pr[:, 0].astype(np.int64) * n + pr[:, 1]).tolist())
    mp = code(G["mp_pairs"])
    pos = {s: code(q[s]["pairs"][q[s]["label"] == 1]) for s in SPLITS}
    assert not (pos["val"] | pos["test"]) & mp, "held-out positives in message passing"
    allpos = set().union(*pos.values())
    for s in SPLITS:
        neg = q[s]["pairs"][q[s]["label"] == 0]
        assert not code(neg) & allpos and (neg[:, 0] != neg[:, 1]).all()
    tfs = {s: set(q[s]["pairs"][:, 0].tolist()) for s in SPLITS}
    assert not (tfs["train"] & tfs["test"]) and not (tfs["train"] & tfs["val"])
    log("  leakage checks passed")


def attach_betas(G, expr_line, P):
    """Fit Eq. 2 for every query pair and message-passing edge; store [b1, b3, t_b3]."""
    genes, n = G["genes"], len(G["genes"])
    allp = np.unique(np.vstack([G["q"][s]["pairs"] for s in SPLITS] + [G["mp_fwd_pairs"]]), axis=0)
    betas = fit_gxp(expr_line, make_Z(expr_line), P,
                    pd.DataFrame({"tf": genes[allp[:, 0]], "target": genes[allp[:, 1]]}))
    gidx = pd.Series(np.arange(n), index=genes)
    lut = pd.DataFrame(betas[["b1", "b3", "t_b3"]].to_numpy(np.float32),
                       index=gidx.loc[betas.tf].to_numpy().astype(np.int64) * n
                       + gidx.loc[betas.target].to_numpy())

    def look(pairs, directed_as_given=True):
        c = pairs[:, 0].astype(np.int64) * n + pairs[:, 1]
        return lut.reindex(c).to_numpy(np.float32)

    out = {s: look(G["q"][s]["pairs"]) for s in SPLITS}
    # reverse mp edges reuse the forward edge's betas
    fwd = np.where(G["mp_dir"][:, None] == 0, G["mp_pairs"], G["mp_pairs"][:, ::-1])
    out["mp"] = look(fwd)
    for k, v in out.items():
        if np.isnan(v).any():
            raise RuntimeError(f"{np.isnan(v).any(1).sum()} pairs in {k} have no regression fit")
    out["p_center"] = betas.attrs["p_center"]
    b1_train = out["train"][:, 0]
    out["e_mean"], out["e_sd"] = float(b1_train.mean()), float(b1_train.std() + 1e-8)
    return out


def e_of_p(beta, p, B, slope_col=1):
    """Standardized e_ij(p) = (b1 + slope*(p - p_center) - mean) / sd. numpy or torch."""
    return (beta[:, 0] + beta[:, slope_col] * (p - B["p_center"]) - B["e_mean"]) / B["e_sd"]

In [8]:
import torch
import torch.nn as nn
import torch.nn.functional as F


def mean_aggregate(msg, dst, n):
    out = torch.zeros(n, msg.size(1), device=msg.device).index_add_(0, dst, msg)
    deg = torch.zeros(n, device=msg.device).index_add_(0, dst, torch.ones_like(dst, dtype=msg.dtype))
    return out / deg.clamp(min=1).unsqueeze(1)


class EdgeConv(nn.Module):
    """h_i' = W_u [h_i || mean_{j->i} MLP([h_j || a_ji])] — messages conditioned on edge features."""
    def __init__(self, d, d_edge):
        super().__init__()
        self.msg = nn.Sequential(nn.Linear(d + d_edge, d), nn.ReLU(), nn.Linear(d, d))
        self.upd = nn.Linear(2 * d, d)

    def forward(self, h, edge_index, edge_attr):
        src, dst = edge_index
        m = self.msg(torch.cat([h[src], edge_attr], dim=1))
        return self.upd(torch.cat([h, mean_aggregate(m, dst, h.size(0))], dim=1))


class DiffGRN(nn.Module):
    """2-layer edge-conditioned message passing + MLP decoder on (h_tf, h_target, e_ij)."""
    def __init__(self, d_in, d=64, dropout=0.3, d_edge=2):
        super().__init__()
        self.inp = nn.Sequential(nn.Dropout(dropout), nn.Linear(d_in, d), nn.ReLU())
        self.conv1, self.conv2 = EdgeConv(d, d_edge), EdgeConv(d, d_edge)
        self.norm1, self.norm2 = nn.LayerNorm(d), nn.LayerNorm(d)
        self.drop = nn.Dropout(dropout)
        self.dec = nn.Sequential(nn.Linear(3 * d + 1, d), nn.ReLU(), nn.Dropout(dropout), nn.Linear(d, 1))

    def encode(self, x, edge_index, edge_attr):
        h = self.inp(x)
        h = self.norm1(h + self.drop(F.relu(self.conv1(h, edge_index, edge_attr))))
        h = self.norm2(h + self.drop(F.relu(self.conv2(h, edge_index, edge_attr))))
        return h

    def decode(self, h, pairs, e):
        hs, ht = h[pairs[:, 0]], h[pairs[:, 1]]
        return self.dec(torch.cat([hs, ht, hs * ht, e.unsqueeze(1)], dim=1)).squeeze(1)

    def forward(self, x, edge_index, edge_attr, pairs, e):
        return self.decode(self.encode(x, edge_index, edge_attr), pairs, e)

In [9]:
from sklearn.metrics import average_precision_score, roc_auc_score

VARIANTS = {
    # name: overrides
    "full": {},
    "mlp_no_graph": {"no_edges": True},          # baseline: no message passing -> per-gene MLP + decoder
    "no_edge_feat": {"use_e": False},            # e_ij zeroed in encoder and decoder
    "b1_only": {"slope": False},                 # e = b1, drop b3*(p - c): Delta = 0 by design
    "t_feature": {"slope_col": 2},               # use t_b3 instead of b3 in Eq. 3
    "is_tf_only": {"x_mode": "is_tf"},           # no expression node features
    "no_is_tf": {"x_mode": "expr"},              # drop the TF flag
    "random_graph": {"rewire": True},            # shuffle targets of message-passing edges
}


def early_precision_ratio(y, s):
    k = int(y.sum())
    top = np.argsort(-s)[:k]
    return float(y[top].mean() / y.mean())


class Runner:
    """Holds tensors for one (condition, seed, variant); trains and scores."""
    def __init__(self, G, B, P_values, variant="full", seed=0):
        self.opt = {"use_e": True, "slope": True, "slope_col": 1, "x_mode": "all", "rewire": False,
                    "no_edges": False}
        self.opt.update(VARIANTS[variant])
        self.dev = torch.device(CFG["device"] if torch.cuda.is_available() else "cpu")
        torch.manual_seed(seed)
        self.rng = np.random.default_rng(seed)
        t = lambda a, dt=torch.float32: torch.as_tensor(np.array(a), dtype=dt, device=self.dev)
        x = G["x"]
        if self.opt["x_mode"] == "is_tf":
            x = np.column_stack([G["is_tf"], np.ones_like(G["is_tf"])])
        elif self.opt["x_mode"] == "expr":
            x = x[:, :-1]
        self.x = t(x)
        mp = G["mp_pairs"].copy()
        if self.opt["rewire"]:
            mp[:, 1] = mp[self.rng.permutation(len(mp)), 1]
        mp_dir, mp_beta = G["mp_dir"], B["mp"]
        if self.opt["no_edges"]:                       # MLP baseline: empty graph, no messages
            mp, mp_dir, mp_beta = mp[:0], mp_dir[:0], mp_beta[:0]
        self.edge_index = t(mp.T.reshape(2, -1), torch.long)
        self.mp_dir = t(mp_dir)
        self.mp_beta = t(mp_beta)
        self.q = {s: {"pairs": t(G["q"][s]["pairs"], torch.long), "label": G["q"][s]["label"],
                      "slot": G["q"][s]["slot"], "beta": t(B[s])} for s in SPLITS}
        self.B, self.P_values = B, np.asarray(P_values)
        self.model = DiffGRN(self.x.size(1), CFG["hidden"], CFG["dropout"]).to(self.dev)

    def e(self, beta, p):
        if not self.opt["use_e"]:
            return torch.zeros(beta.size(0), device=self.dev)
        if not self.opt["slope"]:
            p = self.B["p_center"]
        return e_of_p(beta, p, self.B, self.opt["slope_col"])

    def scores(self, pairs, beta, p, mp_beta=None):
        ea = torch.stack([self.e(self.mp_beta if mp_beta is None else mp_beta, p), self.mp_dir], 1)
        return self.model(self.x, self.edge_index, ea, pairs, self.e(beta, p))

    def evaluate(self, s, p=None):
        self.model.eval()
        p = self.B["p_center"] if p is None else p
        with torch.no_grad():
            sc = self.scores(self.q[s]["pairs"], self.q[s]["beta"], p).cpu().numpy()
        y = self.q[s]["label"]
        return {"auroc": roc_auc_score(y, sc), "auprc": average_precision_score(y, sc),
                "epr": early_precision_ratio(y, sc)}

    def train(self, verbose=False):
        opt = torch.optim.Adam(self.model.parameters(), lr=CFG["lr"], weight_decay=CFG["weight_decay"])
        tr = self.q["train"]
        k = CFG["k_train_negatives"]
        best, best_state, wait, hist = -1, None, 0, []
        for ep in range(CFG["epochs"]):
            self.model.train()
            pick = self.rng.integers(k)                         # one negative slot per epoch
            m = (tr["slot"] == -1) | (tr["slot"] == pick)
            idx = torch.as_tensor(np.where(m)[0], device=self.dev)
            p = float(self.rng.choice(self.P_values)) if CFG["p_mode"] == "sample" else self.B["p_center"]
            logit = self.scores(tr["pairs"][idx], tr["beta"][idx], p)
            loss = F.binary_cross_entropy_with_logits(logit, torch.as_tensor(tr["label"][m], device=self.dev))
            opt.zero_grad()
            loss.backward()
            opt.step()
            v = self.evaluate("val")
            hist.append({"epoch": ep, "loss": loss.item(), **{f"val_{a}": b for a, b in v.items()}})
            if v["auprc"] > best:
                best, wait = v["auprc"], 0
                best_state = {a: b.detach().clone() for a, b in self.model.state_dict().items()}
            else:
                wait += 1
                if wait >= CFG["patience"]:
                    break
            if verbose and ep % 10 == 0:
                print(f"  ep {ep:3d} loss {loss.item():.4f} val AUROC {v['auroc']:.3f} AUPRC {v['auprc']:.3f}")
        self.model.load_state_dict(best_state)
        return pd.DataFrame(hist)

In [10]:
def all_positive_pairs(G):
    return np.vstack([G["q"][s]["pairs"][G["q"][s]["label"] == 1] for s in SPLITS])


def delta_scores(runner, pairs_t, beta_t, mp_beta_t, p_low, p_high):
    runner.model.eval()
    with torch.no_grad():
        lo = torch.sigmoid(runner.scores(pairs_t, beta_t, p_low, mp_beta_t))
        hi = torch.sigmoid(runner.scores(pairs_t, beta_t, p_high, mp_beta_t))
    return (hi - lo).cpu().numpy(), lo.cpu().numpy(), hi.cpu().numpy()


def differential(runner, G, B, expr_line, P, p_low, p_high, seed):
    """Delta for every known edge, plus a pooled permutation null: shuffle P across lines,
    refit Eq. 2, rescore with the SAME trained model (no retraining)."""
    genes, n = G["genes"], len(G["genes"])
    pairs = all_positive_pairs(G)
    dev = runner.dev
    pairs_t = torch.as_tensor(pairs, dtype=torch.long, device=dev)
    beta_all = attach_lookup(G, B, pairs)
    delta, s_lo, s_hi = delta_scores(runner, pairs_t, torch.as_tensor(np.array(beta_all), device=dev),
                                     runner.mp_beta, p_low, p_high)
    rng = np.random.default_rng(seed + 7)
    null = []
    for _ in range(CFG["n_permutations"]):
        Pp = pd.Series(rng.permutation(P.values), index=P.index)
        Bp = attach_betas(G, expr_line, Pp)
        Bp.update({"p_center": B["p_center"], "e_mean": B["e_mean"], "e_sd": B["e_sd"]})
        bp = attach_lookup(G, Bp, pairs)
        d, _, _ = delta_scores(runner, pairs_t, torch.as_tensor(np.array(bp), device=dev),
                               torch.as_tensor(np.array(Bp["mp"]), device=dev), p_low, p_high)
        null.append(np.abs(d))
    null = np.sort(np.concatenate(null))
    pval = (1 + len(null) - np.searchsorted(null, np.abs(delta), side="left")) / (1 + len(null))
    df = pd.DataFrame({"tf": genes[pairs[:, 0]], "target": genes[pairs[:, 1]],
                       "delta": delta, "abs_delta": np.abs(delta),
                       "score_low": s_lo, "score_high": s_hi,
                       "b1": beta_all[:, 0], "b3": beta_all[:, 1], "t_b3": beta_all[:, 2],
                       "p_perm": pval})
    df["q_bh"] = bh(df.p_perm.to_numpy())
    return df.sort_values("abs_delta", ascending=False).reset_index(drop=True), null


def attach_lookup(G, B, pairs):
    """Betas for arbitrary pairs drawn from the query sets (positives of every split)."""
    n = len(G["genes"])
    codes = np.concatenate([G["q"][s]["pairs"][:, 0].astype(np.int64) * n + G["q"][s]["pairs"][:, 1] for s in SPLITS])
    vals = np.vstack([B[s] for s in SPLITS])
    lut = pd.DataFrame(vals, index=codes)
    lut = lut[~lut.index.duplicated()]
    return lut.reindex(pairs[:, 0].astype(np.int64) * n + pairs[:, 1]).to_numpy(np.float32)


def bh(p):
    o = np.argsort(p)
    ranked = p[o] * len(p) / np.arange(1, len(p) + 1)
    q = np.minimum.accumulate(ranked[::-1])[::-1]
    out = np.empty_like(q)
    out[o] = np.minimum(q, 1)
    return out

In [11]:
def rf_differential(expr_line, edges, P, seed=0):
    """GENIE3-style: random forest per target, regressors = TFs, fit separately in the
    bottom and top rf_tail_frac of lines; Delta = importance_high - importance_low."""
    from sklearn.ensemble import RandomForestRegressor
    lo, hi = tail_lines(P, CFG["rf_tail_frac"], seed)
    Xz = (expr_line - expr_line.mean()) / expr_line.std(ddof=0)
    tfs = sorted(edges.tf.unique())
    targets = edges.target.unique()
    if CFG["rf_max_targets"] and len(targets) > CFG["rf_max_targets"]:
        targets = np.random.default_rng(seed).choice(targets, CFG["rf_max_targets"], replace=False)
    rows = []
    t0 = time.time()
    for i, tg in enumerate(targets):
        regs = [t for t in tfs if t != tg]
        imp = {}
        for name, idx in (("low", lo), ("high", hi)):
            rf = RandomForestRegressor(n_estimators=100, max_features="sqrt", random_state=seed, n_jobs=-1)
            rf.fit(Xz.loc[idx, regs].values, Xz.loc[idx, tg].values)
            imp[name] = rf.feature_importances_
        rows.append(pd.DataFrame({"tf": regs, "target": tg, "imp_low": imp["low"], "imp_high": imp["high"]}))
        if i % 200 == 0:
            print(f"  RF target {i}/{len(targets)} ({time.time() - t0:.0f}s)")
    df = pd.concat(rows, ignore_index=True)
    df["delta_rf"] = df.imp_high - df.imp_low
    return df


def compare_rankings(dgnn, other, col, k=1000):
    from scipy.stats import spearmanr
    m = dgnn.merge(other[["tf", "target", col]], on=["tf", "target"])
    k = min(k, max(1, len(m) // 10))           # keep "top" a real subset when few edges are shared
    rho = spearmanr(m.abs_delta, m[col].abs()).correlation
    top_g = set(map(tuple, m.nlargest(k, "abs_delta")[["tf", "target"]].values))
    top_o = set(map(tuple, m.reindex(m[col].abs().sort_values(ascending=False).index)
                    .head(k)[["tf", "target"]].values))
    return {"n_shared": len(m), "k": k, "spearman_abs": float(rho), "top_jaccard": len(top_g & top_o) / len(top_g | top_o)}

In [12]:
def run_all(z_modes=None, conditions=None, seeds=None, variants=None, variant_condition="composite",
            do_differential=True, do_rf=False, verbose=False):
    """Everything: for each seed, one graph; for each z_mode and condition, edge features,
    the logistic floor, all model variants, and Delta for the full model. Results are saved
    after every run, so an interrupted session keeps what finished."""
    z_modes = z_modes or CFG["z_modes"]
    conditions = conditions or CFG["conditions"]
    seeds = seeds or CFG["seeds"]
    variants = variants or list(VARIANTS)
    expr_line, pheno_line, _ = prepare_data()
    edges = drop_untestable_tfs(load_network(expr_line.columns), expr_line)
    edges.to_csv(OUT / "mgrn_mapped.csv", index=False)
    rows, delta_rows = [], []

    def save():
        pd.DataFrame(rows).to_csv(OUT / "linkpred_results.csv", index=False)
        pd.DataFrame(delta_rows).to_csv(OUT / "delta_summary.csv", index=False)
        (OUT / "run_log.txt").write_text("\n".join(LOG))

    for seed in seeds:
        log(f"=== seed {seed}: building graph")
        G = build_graph(expr_line, edges, seed)
        for zm in z_modes:
            CFG["z_mode"] = zm
            for cond in conditions:
                P = get_P(pheno_line, cond)
                p_low, p_high = contrast_values(P, CFG["tail_frac"])
                B = attach_betas(G, expr_line, P)    
                run_variants = variants if cond == variant_condition else ["full"]
                for var in run_variants:
                    t0 = time.time()
                    R = Runner(G, B, P.values, var, seed)
                    hist = R.train(verbose)
                    te = R.evaluate("test")
                    rows.append({"z_mode": zm, "seed": seed, "condition": cond, "variant": var,
                                 "epochs": len(hist), **{f"test_{a}": b for a, b in te.items()},
                                 "secs": time.time() - t0})
                    log(f"  [{zm:13s} | {cond:9s} | {var:12s} | seed {seed}] test AUROC {te['auroc']:.3f} "
                        f"AUPRC {te['auprc']:.3f} EPR {te['epr']:.2f} ({len(hist)} ep, {time.time() - t0:.0f}s)")
                    hist.to_csv(OUT / f"history_{zm}_{cond}_{var}_seed{seed}.csv", index=False)
                    if var == "full" and do_differential:
                        D, null = differential(R, G, B, expr_line, P, p_low, p_high, seed)
                        D.to_csv(OUT / f"delta_{zm}_{cond}_seed{seed}.csv", index=False)
                        np.save(OUT / f"delta_null_{zm}_{cond}_seed{seed}.npy", null)
                        lin = D.assign(delta_lin=D.b3 * (p_high - p_low))
                        cmp = compare_rankings(D, lin, "delta_lin")
                        cut = np.quantile(null, 0.99)
                        n_above = int((D.abs_delta > cut).sum())
                        delta_rows.append({"z_mode": zm, "seed": seed, "condition": cond, "n_edges": len(D),
                                           "q_lt_0.05": int((D.q_bh < 0.05).sum()),
                                           "q_lt_0.10": int((D.q_bh < 0.10).sum()),
                                           "above_null_p99": n_above, "expected_p99": round(0.01 * len(D)),
                                           "median_abs_delta": float(D.abs_delta.median()),
                                           "max_abs_delta": float(D.abs_delta.max()),
                                           "spearman_vs_b3": cmp["spearman_abs"],
                                           "top_jaccard_vs_b3": cmp["top_jaccard"]})
                        log(f"    Delta: {len(D):,} edges; q<0.05: {(D.q_bh < 0.05).sum():,}; "
                            f"above null 99th pct: {n_above:,} (chance ~{round(0.01 * len(D)):,}); "
                            f"vs |b3| Spearman {cmp['spearman_abs']:.3f}")
                        if do_rf and seed == seeds[0]:
                            rf = rf_differential(expr_line, edges, P, seed)
                            rf.to_csv(OUT / f"rf_delta_{zm}_{cond}.csv", index=False)
                            log(f"    vs RF differential: {compare_rankings(D, rf, 'delta_rf')}")
                    save()
    save()
    return pd.DataFrame(rows), pd.DataFrame(delta_rows)

In [13]:
def drop_untestable_tfs(edges, expr_line, min_lines=30):
    """Keep TFs whose locus has >= min_genotyped_lines genotyped lines and varies across lines
    (Eq. 2 can't be fit otherwise). Applied in BOTH modes so the edge set and TF splits
    are identical, making genotype vs expression results directly comparable."""
    saved = CFG["z_mode"]
    CFG["z_mode"] = "genotype"
    Zg = make_Z(expr_line)
    CFG["z_mode"] = saved
    ok = (Zg.notna().sum() >= CFG["min_genotyped_lines"]) & (Zg.std() > 0)
    out = edges[edges.tf.isin(ok[ok].index)].reset_index(drop=True)
    log(f"  testable in genotype mode: kept {out.tf.nunique()} of {edges.tf.nunique()} TFs, "
        f"{len(out):,} edges (used for both modes)")
    return out

In [14]:
VARIANT_LABELS = {
    "mlp_no_graph": "MLP (no graph)", "full": "GNN (full)",
    "no_edge_feat": "GNN, no edge feature", "b1_only": "GNN, no \u03b2\u2083 term",
    "t_feature": "GNN, t(\u03b2\u2083) feature", "is_tf_only": "GNN, no expression",
    "no_is_tf": "GNN, no TF flag", "random_graph": "GNN, shuffled graph",
}


def summarize(res: pd.DataFrame, delta: pd.DataFrame | None = None):
    """Mean +/- SD over seeds of test AUROC/AUPRC/EPR per z_mode x variant, and Delta counts."""
    lp = (res.groupby(["z_mode", "condition", "variant"])[["test_auroc", "test_auprc", "test_epr"]]
          .agg(["mean", "std"]).round(3))
    lp.to_csv(OUT / "linkpred_summary.csv")
    out = {"linkpred": lp}
    if delta is not None and len(delta):
        ds = (delta.groupby(["z_mode", "condition"])
              [["q_lt_0.05", "above_null_p99", "expected_p99", "median_abs_delta", "spearman_vs_b3"]]
              .agg(["mean", "std"]).round(4))
        ds.to_csv(OUT / "delta_summary_by_mode.csv")
        out["delta"] = ds
    return out

In [15]:
# %% 12. SERGIO simulation: ground-truth differential edges ---------------------------------
# Simulates expression for the real RIL lines on a subnetwork of the mGRN leaf network, with a
# known set of DIFFERENTIAL edges whose strength depends on phenotype, then checks whether each
# method recovers them: GNN Delta, and the regression's |t(beta3)| ranking.
#
# Design (SERGIO: Dibaeinia & Sinha 2020, steady-state mode):
#   - Subnetwork: n_tfs random testable TFs and up to n_targets genes regulated by >= 2 of them;
#     TF->TF edges that would create a cycle are dropped (SERGIO needs a DAG).
#   - Lines: the real 214 lines with their real composite phenotype P and real genotypes.
#     Master regulators' production rate in a line = base * (1 + geno_effect * PT fraction at the
#     TF locus), so genotype drives expression through the network.
#   - Differential edges: a random frac_differential of edges have strength
#         K_ij(line) = K_ij * exp(gamma * P_line)
#     SERGIO holds K fixed within a run, so lines are grouped into n_phenotype_groups quantiles of
#     P; each group is one run (its lines are SERGIO "cell types") with K set from the group's mean P.
#   - Bulk expression per line = mean over simulated cells, normalized to CPM, log2(x + 1).
import contextlib
import io
import os
import subprocess
import sys
import tempfile

CFG.update({
    "sim": {
        "enabled": True, "seed": 0, "n_tfs": 60, "n_targets": 400,
        "frac_differential": 0.10, "gamma": 0.5, "geno_effect": 1.0,
        "n_phenotype_groups": 5, "cells_per_line": 20, "n_permutations": 5,
        "seeds": [0, 1, 2], "sergio_dir": None,   # None: ./SERGIO or /kaggle/working/SERGIO
    }
})


def setup_sergio():
    """Import SERGIO, cloning it from GitHub if needed (Kaggle: turn Internet on) and patching
    the numpy aliases (np.int / np.float) removed in numpy >= 1.24."""
    d = CFG["sim"]["sergio_dir"] or ("/kaggle/working/SERGIO" if Path("/kaggle").exists() else "SERGIO")
    if not Path(d, "SERGIO", "sergio.py").exists():
        subprocess.run(["git", "clone", "-q", "https://github.com/PayamDiba/SERGIO.git", d], check=True)
    for fn in ("sergio.py", "gene.py"):
        p = Path(d, "SERGIO", fn)
        src = p.read_text()
        fixed = re.sub(r"np\.int\(", "int(", re.sub(r"np\.float\(", "float(", src))
        fixed = re.sub(r"dtype\s*=\s*np\.int\b", "dtype=int", fixed)
        if fixed != src:
            p.write_text(fixed)
    if d not in sys.path:
        sys.path.insert(0, d)
    from SERGIO.sergio import sergio
    return sergio


def sample_subnetwork(edges, rng, n_tfs, n_targets):
    """n_tfs random TFs; targets regulated by >= 2 of them (so targets combine inputs), up to
    n_targets of those; made acyclic by a random TF order (SERGIO needs a DAG)."""
    tfs = rng.choice(edges.tf.unique(), size=min(n_tfs, edges.tf.nunique()), replace=False)
    sub = edges[edges.tf.isin(tfs)]
    order = {tf: i for i, tf in enumerate(rng.permutation(tfs))}
    tgt_order = sub.target.map(order)
    sub = sub[(tgt_order.isna() | (tgt_order > sub.tf.map(order))) & (sub.tf != sub.target)]
    indeg = sub[~sub.target.isin(tfs)].target.value_counts()
    multi = indeg.index[indeg >= 2].to_numpy()
    chosen = set(rng.choice(multi, size=min(n_targets, len(multi)), replace=False)) | set(tfs)
    return sub[sub.target.isin(chosen)].reset_index(drop=True)


def simulate_sergio(edges, P, Zm_lines, gene2marker):
    """Returns (expr_line_sim [lines x genes], sim_edges [tf, target, K, differential])."""
    S = CFG["sim"]
    sergio = setup_sergio()
    rng = np.random.default_rng(S["seed"])
    sub = sample_subnetwork(edges, rng, S["n_tfs"], S["n_targets"])
    genes = pd.Index(sorted(set(sub.tf) | set(sub.target)))
    gid = pd.Series(np.arange(len(genes)), index=genes)
    sub["K"] = rng.uniform(1, 4, len(sub)) * rng.choice([1, -1], len(sub), p=[0.7, 0.3])
    sub["differential"] = rng.random(len(sub)) < S["frac_differential"]
    masters = [g for g in genes if g not in set(sub.target)]
    base = pd.Series(rng.uniform(0.5, 2.0, len(masters)), index=masters)
    log(f"SERGIO subnetwork: {len(genes)} genes, {sub.tf.nunique()} TFs ({len(masters)} master regulators), "
        f"{len(sub)} edges, {int(sub.differential.sum())} differential")

    lines = P.index
    geno = pd.DataFrame(index=lines)
    for m in masters:                                 # PT fraction at each master regulator's locus
        mk = gene2marker.get(m)
        geno[m] = Zm_lines[mk].reindex(lines).fillna(0).values if pd.notna(mk) and mk in Zm_lines else 0.0
    groups = pd.qcut(P.rank(method="first"), S["n_phenotype_groups"], labels=False)
    expr = pd.DataFrame(index=lines, columns=genes, dtype=float)
    tmp = tempfile.mkdtemp()
    for gidx in range(S["n_phenotype_groups"]):
        gl = lines[groups == gidx]
        scale = np.where(sub.differential, np.exp(S["gamma"] * P.loc[gl].mean()), 1.0)
        Kg = sub.K.values * scale
        rows = []
        for tgt, grp in sub.assign(Kg=Kg).groupby("target"):
            regs = gid[grp.tf].tolist()
            rows.append([gid[tgt], len(regs), *regs, *grp.Kg.tolist(), *([2.0] * len(regs))])
        ft, fr = os.path.join(tmp, f"targets_{gidx}.txt"), os.path.join(tmp, f"regs_{gidx}.txt")
        Path(ft).write_text("\n".join(",".join(map(str, r)) for r in rows))
        rates = [[gid[m], *(base[m] * (1 + S["geno_effect"] * geno.loc[gl, m].values))] for m in masters]
        Path(fr).write_text("\n".join(",".join(map(str, r)) for r in rates))
        sim = sergio(number_genes=len(genes), number_bins=len(gl), number_sc=S["cells_per_line"],
                     noise_params=1, decays=0.8, sampling_state=15, noise_type="dpd")
        with contextlib.redirect_stdout(io.StringIO()):
            sim.build_graph(input_file_taregts=ft, input_file_regs=fr, shared_coop_state=2)
            sim.simulate()
        e = sim.getExpressions()                      # bins x genes x cells
        expr.loc[gl] = e.mean(axis=2)
        log(f"  SERGIO group {gidx + 1}/{S['n_phenotype_groups']}: {len(gl)} lines, mean P {P.loc[gl].mean():+.2f}")
    cpm = expr.div(expr.sum(axis=1), axis=0) * 1e6
    return np.log2(cpm.astype(float) + 1), sub


def detection_metrics(D, truth):
    """How well |Delta| and |t(beta3)| rank the true differential edges among all edges."""
    m = D.merge(truth[["tf", "target", "differential"]], on=["tf", "target"])
    y = m.differential.astype(int).values
    out = {"n_edges": len(m), "n_differential": int(y.sum())}
    for name, score in (("delta", m.abs_delta), ("abs_t_b3", m.t_b3.abs())):
        out[f"{name}_auroc"] = roc_auc_score(y, score)
        out[f"{name}_auprc"] = average_precision_score(y, score)
    out["auprc_chance"] = y.mean()
    out["delta_q05_hits"] = int(((m.q_bh < 0.05) & (m.differential)).sum())
    out["delta_q05_total"] = int((m.q_bh < 0.05).sum())
    return out


def run_simulation(z_modes=None, seeds=None):
    """Simulate once, then train the full GNN per seed x z_mode and score detection of the
    planted differential edges. Saves sim_results.csv."""
    S = CFG["sim"]
    z_modes, seeds = z_modes or CFG["z_modes"], seeds or S["seeds"]
    expr_line, pheno_line, _ = prepare_data()
    edges = drop_untestable_tfs(load_network(expr_line.columns), expr_line)
    P = get_P(pheno_line, "composite")
    expr_sim, truth = simulate_sergio(edges, P, GENO["Z_markers"], GENO["gene2marker"])
    expr_sim.to_csv(OUT / "sim_expression.csv")
    truth.to_csv(OUT / "sim_network_truth.csv", index=False)
    p_low, p_high = contrast_values(P, CFG["tail_frac"])
    saved_perm = CFG["n_permutations"]
    CFG["n_permutations"] = S["n_permutations"]
    rows = []
    try:
        for seed in seeds:
            G = build_graph(expr_sim, truth[["tf", "target"]], seed)
            for zm in z_modes:
                CFG["z_mode"] = zm
                B = attach_betas(G, expr_sim, P)
                R = Runner(G, B, P.values, "full", seed)
                R.train()
                te = R.evaluate("test")
                D, _ = differential(R, G, B, expr_sim, P, p_low, p_high, seed)
                D.to_csv(OUT / f"sim_delta_{zm}_seed{seed}.csv", index=False)
                det = detection_metrics(D, truth)
                rows.append({"z_mode": zm, "seed": seed, "linkpred_test_auroc": te["auroc"],
                             "linkpred_test_auprc": te["auprc"], **det})
                log(f"  [SIM {zm:13s} | seed {seed}] link AUROC {te['auroc']:.3f} | differential-edge AUROC: "
                    f"Delta {det['delta_auroc']:.3f}, |t(b3)| {det['abs_t_b3_auroc']:.3f} | "
                    f"AUPRC: Delta {det['delta_auprc']:.3f}, |t(b3)| {det['abs_t_b3_auprc']:.3f} "
                    f"(chance {det['auprc_chance']:.3f})")
                pd.DataFrame(rows).to_csv(OUT / "sim_results.csv", index=False)
    finally:
        CFG["n_permutations"] = saved_perm
    res = pd.DataFrame(rows)
    summ = res.groupby("z_mode").agg(["mean", "std"]).round(3)
    summ.to_csv(OUT / "sim_summary.csv")
    return res, summ

In [16]:
if __name__ == "__main__":
    res, delta = run_all(verbose=False)       # do_rf=True adds the random-forest Delta baseline (slow)
    s = summarize(res, delta)
    print(s["linkpred"])
    print(s.get("delta"))
    if CFG["sim"]["enabled"]:
        sim_res, sim_summary = run_simulation()
        print(sim_summary)

Raw counts: 375 plants x 15346 genes
Genotypes: 322 samples x 23706 marker genes; missing 0.3%
Sample labels: 320 from genotype key (85 differ from RNA metadata, mostly plate PN3); 43 from metadata; 12 PN3 samples dropped (unverifiable)
RIL plants kept: 288 from 214 lines
Genotype matrix: 201 of 214 lines genotyped; mean PT fraction 0.218 (BC1S5 expectation 0.25)
Genes mapped to markers: 10720 are markers, 4539 use nearest marker, 2 unmapped
Genes kept: 15261; line-level expression (214, 15261)
Line-level trait Spearman:
       antho  necro  wilt
antho   1.00  -0.17  0.27
necro  -0.17   1.00  0.06
wilt    0.27   0.06  1.00
ID map: 27,149 strict 1:1 v4->v5 pairs
mGRN leaf: 313,591 edges, 1,100 TFs
  after v4->v5: 228,817 edges, 949 TFs
  after expressed-gene filter: 96,263 edges, 423 TFs, 10,985 targets
  testable in genotype mode: kept 417 of 423 TFs, 95,203 edges (used for both modes)
=== seed 0: building graph
  train: 285 TFs, 66,072 positives, 330,360 negatives
  val: 45 TFs, 9,648